# AX630C Plate+Face Demo v2.5-rc — Google Colab Pipeline (Drive-Persistent)

- Drive Source Folder: `https://drive.google.com/drive/folders/1LGgWu-NEop0IMkgBojNCJyCg027siA8r?usp=drive_link`
- Persistent Output Dir on Drive: `/content/drive/MyDrive/ax630c_v25_rc_persist/`
- Every stage writes logs, metrics, and `.axmodel` artifacts to `PERSIST_DIR` so nothing is lost on disconnect.

In [ ]:
# === Cell 1: Mount Google Drive, Set Up Persistence, Locate/Download Folder ===
from google.colab import drive
drive.mount('/content/drive')

import os, glob, subprocess, shutil

FOLDER_ID = '1LGgWu-NEop0IMkgBojNCJyCg027siA8r'
FOLDER_URL = f'https://drive.google.com/drive/folders/{FOLDER_ID}?usp=drive_link'
PERSIST_DIR = '/content/drive/MyDrive/ax630c_v25_rc_persist'
WORK_DIR = '/content/v25_work'
for d in [PERSIST_DIR, WORK_DIR, f'{PERSIST_DIR}/logs', f'{PERSIST_DIR}/models', f'{PERSIST_DIR}/metrics']:
    os.makedirs(d, exist_ok=True)

!pip -q install -U gdown onnx onnxruntime numpy opencv-python-headless

# 优先查找 MyDrive 中已存在的 'grok bot 文件内容' 目录，避免重复下载
ASSETS_DIR = None
for cand in glob.glob('/content/drive/MyDrive/**/SHA256SUMS.txt', recursive=True):
    parent = os.path.dirname(cand)
    if os.path.isdir(os.path.join(parent, 'PULSAR2')) and os.path.isdir(os.path.join(parent, 'B_COLOR')):
        ASSETS_DIR = parent
        break

if ASSETS_DIR:
    print('Found mounted Drive assets directly at:', ASSETS_DIR)
else:
    ASSETS_DIR = f'{PERSIST_DIR}/assets_cache'
    os.makedirs(ASSETS_DIR, exist_ok=True)
    if not os.path.exists(os.path.join(ASSETS_DIR, 'SHA256SUMS.txt')):
        print('Downloading shared Drive folder into:', ASSETS_DIR)
        !gdown --folder "{FOLDER_URL}" -O "{ASSETS_DIR}" --remaining-ok
    else:
        print('Using existing cached assets in:', ASSETS_DIR)

with open(f'{PERSIST_DIR}/assets_dir.txt', 'w') as f:
    f.write(ASSETS_DIR)

print('=== ASSETS TREE ===')
for root, dirs, files in os.walk(ASSETS_DIR):
    rel = os.path.relpath(root, ASSETS_DIR)
    depth = 0 if rel == '.' else rel.count(os.sep) + 1
    if depth <= 3:
        for f in sorted(files):
            p = os.path.join(root, f)
            sz = os.path.getsize(p)
            print(f'{os.path.relpath(p, ASSETS_DIR)}  ({sz:,} B)')

for doc in ['TRANSFER_STATUS.md', 'README_TRANSFER.md']:
    dp = os.path.join(ASSETS_DIR, doc)
    if os.path.exists(dp):
        print(f'\n=== {doc} ===')
        print(open(dp, encoding='utf-8', errors='replace').read()[:4000])

In [ ]:
# === Cell 2: Clone Repo & Align v2.5-rc Canonical ONNX ===
import hashlib
REPO_DIR = '/content/ax630c-plate-face-demo'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/dehai168/ax630c-plate-face-demo.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch --all --tags
!git checkout -B v2.5-rc origin/cursor/v25-c-integration-audit-50d6
!git merge origin/cursor/mixed-finetune-onnx-b3c5 --no-edit

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()

rec_sha = sha256('models/onnx/plate_rec_finetune.onnx')
col_sha = sha256('models/onnx/plate_color_finetune.onnx')
print('REC   SHA256:', rec_sha)
print('COLOR SHA256:', col_sha)
assert rec_sha == '45b005b56e92df83bbba2199e97935c7897ff23996905d6e6eed0b3ea3af4ae1'
assert col_sha == 'ecba573c49a847321def9106b25d6db756fd98ddd98c0909007d074e4f104360'
print('PASS: Canonical ONNX SHA256 verified!')

In [ ]:
# === Cell 3: Write Exact Mathematical CLE Helper (tools/color_v2_cle.py) ===
%%writefile /content/ax630c-plate-face-demo/tools/color_v2_cle.py
import argparse, hashlib, os, sys
import numpy as np
import onnx
from onnx import numpy_helper

CANONICAL_SHA256 = 'ecba573c49a847321def9106b25d6db756fd98ddd98c0909007d074e4f104360'

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()

def apply_cle(model, iters=20, eps=1e-6, scale_clip=10.0):
    inits = {i.name: i for i in model.graph.initializer}
    chain = [n for n in model.graph.node if n.op_type in ('Conv', 'Gemm')]
    weights = [numpy_helper.to_array(inits[n.input[1]]).astype(np.float64).copy() for n in chain]
    biases = [numpy_helper.to_array(inits[n.input[2]]).astype(np.float64).copy() if len(n.input) > 2 else None for n in chain]
    for _ in range(iters):
        for i in range(len(chain) - 1):
            w1, b1, w2, n2 = weights[i], biases[i], weights[i + 1], chain[i + 1]
            r1 = np.max(np.abs(w1), axis=(1, 2, 3))
            if n2.op_type == 'Conv':
                r2 = np.max(np.abs(w2), axis=(0, 2, 3))
            else:
                tb = next((a.i for a in n2.attribute if a.name == 'transB'), 0)
                r2 = np.max(np.abs(w2), axis=0 if tb else 1)
            valid = (r1 > eps) & (r2 > eps)
            s = np.ones_like(r1)
            s[valid] = np.clip(np.sqrt(r2[valid] / r1[valid]), 1.0 / scale_clip, scale_clip)
            weights[i] = w1 * s[:, None, None, None]
            if b1 is not None: biases[i] = b1 * s
            if n2.op_type == 'Conv':
                weights[i + 1] = w2 / s[None, :, None, None]
            else:
                tb = next((a.i for a in n2.attribute if a.name == 'transB'), 0)
                weights[i + 1] = w2 / (s[None, :] if tb else s[:, None])
    for i, n in enumerate(chain):
        inits[n.input[1]].CopyFrom(numpy_helper.from_array(weights[i].astype(np.float32), name=n.input[1]))
        if biases[i] is not None:
            inits[n.input[2]].CopyFrom(numpy_helper.from_array(biases[i].astype(np.float32), name=n.input[2]))
    return model

def verify_equiv(orig_path, cle_path, n_rand=32, tol=1e-4):
    import onnxruntime as ort
    s0 = ort.InferenceSession(orig_path, providers=['CPUExecutionProvider'])
    s1 = ort.InferenceSession(cle_path, providers=['CPUExecutionProvider'])
    inp = s0.get_inputs()[0].name
    xs = np.random.default_rng(20261124).uniform(0.0, 1.0, (n_rand, 3, 96, 96)).astype(np.float32)
    md, ms, am = 0.0, 0.0, 0
    for i in range(n_rand):
        y0 = s0.run(None, {inp: xs[i:i+1]})[0]
        y1 = s1.run(None, {inp: xs[i:i+1]})[0]
        d = np.abs(y0 - y1)
        md = max(md, float(d.max())); ms += float(d.mean())
        am += int(np.argmax(y0) == np.argmax(y1))
    print(f'[CLE verify] max_diff={md:.3e} mae={ms/n_rand:.3e} argmax={am}/{n_rand}')
    assert md < tol and am == n_rand

if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--in', dest='inp', required=True)
    ap.add_argument('--out', required=True)
    args = ap.parse_args()
    assert sha256_file(args.inp) == CANONICAL_SHA256
    m = apply_cle(onnx.load(args.inp))
    onnx.save(m, args.out)
    verify_equiv(args.inp, args.out)
    print('[CLE] PASS ->', args.out, sha256_file(args.out))
